# Step 8: AI Enrichment (Deep) - Synapse Edition
Urgency scoring and complexity analysis. **Environment: Azure Synapse Analytics**.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

storage_account = "fileuploadsa35b5"
input_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/cleaned_data"
output_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/enriched_data"
df = spark.read.format("delta").load(input_path)

# Enrichment Logic
df = df.withColumn("urgency_score", F.least(F.lit(1.0), (F.when(F.col("description").rlike("(?i)urgent|critical"), 0.5).otherwise(0.0) + F.when(F.col("amount") > 10000, 0.3).otherwise(0.0))))

# DISPLAY HIGH URGENCY INSIGHTS
display(df.filter(F.col("urgency_score") > 0.4).limit(10))

df.write.format("delta").mode("overwrite").save(output_path)